In [1]:
# Import libraries
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from scipy.cluster.hierarchy import dendrogram, linkage

from sklearn.base import clone
from sklearn.model_selection import (
    train_test_split, GridSearchCV, StratifiedKFold, KFold,
    cross_validate, cross_val_predict, validation_curve
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor
)
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    roc_curve, classification_report, confusion_matrix, ConfusionMatrixDisplay,
    mean_absolute_error, mean_squared_error, r2_score,
    silhouette_score, adjusted_rand_score
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")

# Standard seed used throughout the notebook for reproducibility
SEED = 42

In [ ]:
# Load dataset
file_path = r'C:\Users\Lenovo\OneDrive\Dokumente\New learning\Ironhack\Final project\clean_employee_data.csv'
employee_df = pd.read_csv(file_path)
employee_df.info()

In [ ]:
#Describtion
employee_df.describe()

In [ ]:
# Keep Attrition
employee_df["Attrition"] = employee_df["Attrition"].map({
    "Yes": 1,
    "No": 0
}).fillna(employee_df["Attrition"]).astype(int)

employee_df["Attrition"].value_counts()

In [ ]:
#Percentage
attrition_counts = employee_df["Attrition"].value_counts()
print(attrition_counts)
attrition_percentage = employee_df["Attrition"].value_counts(normalize=True) * 100

print("\nPercentage:")
print(attrition_percentage)

In [ ]:
#Number of Employees
plt.figure(figsize=(7, 5))

sns.countplot(
    data= employee_df,
    x="Attrition"
)

plt.title("Employee Attrition Distribution")
plt.xlabel("Attrition (0 = No, 1 = Yes)")
plt.ylabel("Number of Employees")

plt.show()

In [ ]:
#Overtime
plt.figure(figsize=(8, 5))

sns.countplot(
    data=employee_df,
    x="OverTime",
    hue="Attrition"
)

plt.title("Employee Attrition by Overtime")
plt.xlabel("Overtime")
plt.ylabel("Number of Employees")

plt.show()

In [ ]:
#Job Role
plt.figure(figsize=(12, 6))

sns.countplot(
    data=employee_df,
    x="JobRole",
    hue="Attrition"
)

plt.title("Employee Attrition by Job Role")
plt.xticks(rotation=45)
plt.xlabel("Job Role")
plt.ylabel("Number of Employees")

plt.show()

In [ ]:
#Age Distribution
plt.figure(figsize=(9, 5))

sns.histplot(
    data=employee_df,
    x="Age",
    hue="Attrition",
    kde=True,
    bins=20
)

plt.title("Age Distribution by Attrition")
plt.xlabel("Age")
plt.ylabel("Number of Employees")

plt.show()

In [ ]:
#Monthly Income
plt.figure(figsize=(9, 5))

sns.boxplot(
    data=employee_df,
    x="Attrition",
    y="MonthlyIncome"
)
plt.title("Monthly Income by Attrition")
plt.xlabel("Attrition")
plt.ylabel("Monthly Income")

plt.show()

In [ ]:
#Year at company
plt.figure(figsize=(9, 5))

sns.boxplot(
    data=employee_df,
    x="Attrition",
    y="YearsAtCompany"
)
plt.title("Years at Company by Attrition")
plt.xlabel("Attrition")
plt.ylabel("Years at Company")

plt.show()

In [ ]:
X = employee_df.drop("Attrition", axis=1)
y = employee_df["Attrition"]

print("Features:", X.shape)
print("Target:", y.shape)

In [ ]:
# Feature types
categorical_features = [
    "BusinessTravel", "Department", "EducationField", "JobRole", "MaritalStatus"
]
numeric_features = [c for c in X.columns if c not in categorical_features]

print("Numerical features:", numeric_features)
print("\nCategorical features:", categorical_features)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

In [ ]:
#Feature preprocessing
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

try:
    categorical_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
except TypeError:
    categorical_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", categorical_encoder)
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
#Baseline Logistic Regression
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000
            )
        )
    ]
)

logistic_pipeline.fit(X_train, y_train)

y_pred = logistic_pipeline.predict(X_test)
y_probability = logistic_pipeline.predict_proba(X_test)[:, 1]

In [ ]:
#Evaluate baseline model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_probability)

print("Baseline Logistic Regression")
print("--------------------------------")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC : {roc_auc:.4f}")

In [ ]:
#Confusion matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Logistic Regression Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
#Compare multiple classification models
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "KNN": KNeighborsClassifier(),

    "SVM": SVC(
        probability=True,
        random_state=42
    ),

    "Naive Bayes": GaussianNB(),

    "Random Forest": RandomForestClassifier(
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    )
}

In [ ]:
#Train and evaluate all models
results = []

for name, model in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", model)
        ]
    )

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(
            y_test,
            predictions
        ),
        "Precision": precision_score(
            y_test,
            predictions
        ),
        "Recall": recall_score(
            y_test,
            predictions
        ),
        "F1": f1_score(
            y_test,
            predictions
        ),
        "ROC_AUC": roc_auc_score(
            y_test,
            probabilities
        )
    })

results_df = pd.DataFrame(results)

display(
    results_df.sort_values(
        by="F1",
        ascending=False
    )

)

In [ ]:
#Visualize model performance
plt.figure(figsize=(12, 6))

sns.barplot(
    data=results_df,
    x="Model",
    y="F1"
)

plt.title("Classification Model F1 Scores")
plt.xticks(rotation=45)
plt.ylabel("F1 Score")

plt.show()

In [ ]:
#Handle class imbalance with SMOTE
try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE
except ImportError:
    raise ImportError("Install imbalanced-learn with: pip install imbalanced-learn")

smote_pipeline = ImbPipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("classifier", LogisticRegression(max_iter=2000))
    ]
)

smote_pipeline.fit(X_train, y_train)

smote_predictions = smote_pipeline.predict(X_test)
smote_probabilities = smote_pipeline.predict_proba(X_test)[:, 1]


In [ ]:
#Execution
print("Logistic Regression + SMOTE")

print(
    "Accuracy:",
    accuracy_score(
        y_test,
        smote_predictions
    )
)

print(
    "Precision:",
    precision_score(
        y_test,
        smote_predictions
    )
)

print(
    "Recall:",
    recall_score(
        y_test,
        smote_predictions
    )
)

print(
    "F1:",
    f1_score(
        y_test,
        smote_predictions
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_test,
        smote_probabilities
    )
)

## 4-5. Classification models & class-imbalance handling
Every model is trained three ways so before/after can be compared:
**Baseline**, **Class weights** (only models that support them) and **SMOTE**
(applied inside an imblearn pipeline, so it only ever touches training data).

In [ ]:
def make_models(balanced=False):
    """Fresh model instances. balanced=True keeps only models with class_weight."""
    cw = "balanced" if balanced else None
    models = {
        "Logistic Regression": LogisticRegression(max_iter=2000, class_weight=cw),
        "Decision Tree": DecisionTreeClassifier(random_state=SEED, class_weight=cw),
        "SVM": SVC(probability=True, random_state=SEED, class_weight=cw),
        "Random Forest": RandomForestClassifier(random_state=SEED, class_weight=cw),
    }
    if not balanced:   # these have no class_weight parameter
        models["KNN"] = KNeighborsClassifier()
        models["Naive Bayes"] = GaussianNB()
        models["Gradient Boosting"] = GradientBoostingClassifier(random_state=SEED)
    return models


def evaluate(pipe, X_tr, y_tr, X_te, y_te):
    """Fit on train, score on test."""
    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_te)
    proba = pipe.predict_proba(X_te)[:, 1]
    return {
        "Accuracy": accuracy_score(y_te, pred),
        "Precision": precision_score(y_te, pred, zero_division=0),
        "Recall": recall_score(y_te, pred),
        "F1": f1_score(y_te, pred),
        "ROC_AUC": roc_auc_score(y_te, proba),
    }


def plain_pipe(model):
    return Pipeline([("preprocessor", preprocessor), ("classifier", model)])


def smote_pipe(model):
    return ImbPipeline([
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=SEED)),
        ("classifier", model),
    ])

In [ ]:
rows = []
for name, model in make_models().items():
    rows.append({"Strategy": "Baseline", "Model": name,
                 **evaluate(plain_pipe(model), X_train, y_train, X_test, y_test)})
for name, model in make_models(balanced=True).items():
    rows.append({"Strategy": "Class weights", "Model": name,
                 **evaluate(plain_pipe(model), X_train, y_train, X_test, y_test)})
for name, model in make_models().items():
    rows.append({"Strategy": "SMOTE", "Model": name,
                 **evaluate(smote_pipe(model), X_train, y_train, X_test, y_test)})

imbalance_df = pd.DataFrame(rows)

display(imbalance_df.pivot(index="Model", columns="Strategy", values="Recall")
        .round(3).style.set_caption("Recall by imbalance strategy"))
display(imbalance_df.pivot(index="Model", columns="Strategy", values="F1")
        .round(3).style.set_caption("F1 by imbalance strategy"))
display(imbalance_df.pivot(index="Model", columns="Strategy", values="ROC_AUC")
        .round(3).style.set_caption("ROC-AUC by imbalance strategy"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, metric in zip(axes, ["Recall", "F1"]):
    sns.barplot(data=imbalance_df, x="Model", y=metric, hue="Strategy", ax=ax)
    ax.set_title(f"{metric}: before vs after imbalance handling")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

> **How to read this:** accuracy is misleading here (~84% of employees stay, so
> always predicting "stay" scores ~84%). Focus on **Recall / F1 / ROC-AUC** for the
> attrition class. Expect recall to rise with class weights / SMOTE while precision drops.

## 6. 5-fold Stratified Cross-Validation
Run on the **training set only**, so the test set stays untouched until the final evaluation.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

cv_rows = []
for strategy, builder in [("Baseline", plain_pipe), ("SMOTE", smote_pipe)]:
    for name, model in make_models().items():
        res = cross_validate(builder(model), X_train, y_train, cv=cv,
                             scoring=scoring, n_jobs=-1)
        row = {"Strategy": strategy, "Model": name}
        for m in scoring:
            row[f"{m}_mean"] = res[f"test_{m}"].mean()
            row[f"{m}_std"] = res[f"test_{m}"].std()
        cv_rows.append(row)

cv_df = pd.DataFrame(cv_rows)
display(cv_df.sort_values("f1_mean", ascending=False).round(3))

## 6b. Hyperparameter tuning (GridSearchCV) & bias-variance
Random Forest, Decision Tree, SVM and KNN are tuned with 5-fold stratified CV,
optimising F1. SMOTE sits inside the pipeline, so it is re-fitted on each
training fold only (no leakage into the validation fold).

In [ ]:
search_space = {
    "Random Forest": (
        RandomForestClassifier(random_state=SEED),
        {"classifier__n_estimators": [200, 400],
         "classifier__max_depth": [None, 5, 10],
         "classifier__min_samples_leaf": [1, 2, 4]},
    ),
    "Decision Tree": (
        DecisionTreeClassifier(random_state=SEED),
        {"classifier__max_depth": [3, 5, 7, 10, None],
         "classifier__min_samples_leaf": [1, 5, 10, 20],
         "classifier__criterion": ["gini", "entropy"]},
    ),
    "SVM": (
        SVC(probability=True, random_state=SEED),
        {"classifier__C": [0.1, 1, 10],
         "classifier__kernel": ["rbf", "linear"],
         "classifier__gamma": ["scale", 0.01, 0.001]},
    ),
    "KNN": (
        KNeighborsClassifier(),
        {"classifier__n_neighbors": [3, 5, 7, 11, 15, 21],
         "classifier__weights": ["uniform", "distance"],
         "classifier__p": [1, 2]},
    ),
}

tuned_models, tuned_rows = {}, []
for name, (model, grid) in search_space.items():
    gs = GridSearchCV(smote_pipe(model), param_grid=grid, cv=cv, scoring="f1",
                      n_jobs=-1, return_train_score=True)
    gs.fit(X_train, y_train)
    tuned_models[name] = gs.best_estimator_

    best = gs.best_index_
    train_f1 = gs.cv_results_["mean_train_score"][best]
    cv_f1 = gs.best_score_
    test_metrics = evaluate(clone(gs.best_estimator_), X_train, y_train, X_test, y_test)

    tuned_rows.append({
        "Model": name,
        "Best_Params": {k.replace("classifier__", ""): v for k, v in gs.best_params_.items()},
        "CV_F1": cv_f1,
        "Train_F1": train_f1,
        "Train_minus_CV_gap": train_f1 - cv_f1,     # large gap = high variance / overfitting
        **{f"Test_{k}": v for k, v in test_metrics.items()},
    })
    print(f"{name:14s} best CV F1 = {cv_f1:.3f} | params = {gs.best_params_}")

tuned_df = pd.DataFrame(tuned_rows).sort_values("CV_F1", ascending=False)
display(tuned_df.drop(columns="Best_Params").round(3))

In [ ]:
# Bias-variance illustration: Decision Tree depth vs train / CV F1
depths = np.arange(1, 16)
train_sc, val_sc = validation_curve(
    smote_pipe(DecisionTreeClassifier(random_state=SEED)),
    X_train, y_train,
    param_name="classifier__max_depth", param_range=depths,
    cv=cv, scoring="f1", n_jobs=-1
)

plt.figure(figsize=(9, 5))
plt.plot(depths, train_sc.mean(axis=1), "o-", label="Training F1")
plt.plot(depths, val_sc.mean(axis=1), "o-", label="Cross-validation F1")
plt.fill_between(depths, val_sc.mean(1) - val_sc.std(1), val_sc.mean(1) + val_sc.std(1), alpha=0.15)
plt.xlabel("Decision Tree max_depth (model complexity)")
plt.ylabel("F1 score")
plt.title("Bias-variance trade-off\nshallow = high bias (underfit) | deep = high variance (overfit)")
plt.legend()
plt.show()

## Final model selection & evaluation
The champion is chosen by **cross-validated F1** (not by test score), then
evaluated once on the untouched test set.

In [ ]:
best_name = tuned_df.iloc[0]["Model"]
final_model = tuned_models[best_name]
print("Selected model:", best_name)

final_pred = final_model.predict(X_test)
final_proba = final_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, final_pred, target_names=["Stay", "Leave"]))
print("ROC-AUC:", round(roc_auc_score(y_test, final_proba), 4))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay(confusion_matrix(y_test, final_pred),
                       display_labels=["Stay", "Leave"]).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title(f"{best_name} - Confusion Matrix (test)")

for name, mdl in tuned_models.items():
    fpr, tpr, _ = roc_curve(y_test, mdl.predict_proba(X_test)[:, 1])
    auc = roc_auc_score(y_test, mdl.predict_proba(X_test)[:, 1])
    axes[1].plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC curves - tuned models (test)")
axes[1].legend()
plt.tight_layout()
plt.show()

## 7. Regression - predicting MonthlyIncome
`Attrition` is excluded from the features. Note that `JobLevel` is very strongly
tied to income, so expect a high R² driven largely by that one feature.

In [ ]:
X_reg = employee_df.drop(columns=["MonthlyIncome", "Attrition"])
y_reg = employee_df["MonthlyIncome"]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=SEED
)

reg_categorical = [c for c in categorical_features if c in X_reg.columns]
reg_numeric = [c for c in X_reg.columns if c not in reg_categorical]

reg_preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]), reg_numeric),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
     reg_categorical),
])

reg_models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regression": DecisionTreeRegressor(max_depth=6, min_samples_leaf=10, random_state=SEED),
    "Random Forest Regression": RandomForestRegressor(n_estimators=300, min_samples_leaf=2,
                                                      random_state=SEED, n_jobs=-1),
}

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
reg_rows, reg_fitted = [], {}
for name, model in reg_models.items():
    pipe = Pipeline([("preprocessor", reg_preprocessor), ("regressor", model)])
    pipe.fit(Xr_train, yr_train)
    pred = pipe.predict(Xr_test)
    cv_scores = cross_validate(clone(pipe), Xr_train, yr_train, cv=kf, scoring="r2")["test_score"]
    reg_fitted[name] = pipe
    reg_rows.append({
        "Model": name,
        "MAE": mean_absolute_error(yr_test, pred),
        "RMSE": np.sqrt(mean_squared_error(yr_test, pred)),
        "R2": r2_score(yr_test, pred),
        "CV_R2_mean": cv_scores.mean(),
    })

regression_df = pd.DataFrame(reg_rows).sort_values("R2", ascending=False)
display(regression_df.round(3))

In [ ]:
best_reg_name = regression_df.iloc[0]["Model"]
best_reg = reg_fitted[best_reg_name]
pred_reg = best_reg.predict(Xr_test)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].scatter(yr_test, pred_reg, alpha=0.5)
lims = [yr_test.min(), yr_test.max()]
axes[0].plot(lims, lims, "r--")
axes[0].set_xlabel("Actual MonthlyIncome")
axes[0].set_ylabel("Predicted MonthlyIncome")
axes[0].set_title(f"{best_reg_name}: actual vs predicted")

rf_pipe = reg_fitted["Random Forest Regression"]
names = rf_pipe.named_steps["preprocessor"].get_feature_names_out()
imp = pd.Series(rf_pipe.named_steps["regressor"].feature_importances_, index=names)
imp.index = [n.split("__", 1)[1] for n in imp.index]
imp.nlargest(10).sort_values().plot.barh(ax=axes[1])
axes[1].set_title("Random Forest - top 10 features for MonthlyIncome")
plt.tight_layout()
plt.show()

## 8. Unsupervised learning - K-Means, Hierarchical Clustering, PCA
Attrition is removed, so the clusters are found without knowing who left.
Attrition is only used afterwards to *describe* each cluster.

In [ ]:
# Same preprocessing (scale numeric + one-hot categorical), target already excluded (X has no Attrition)
X_cluster = clone(preprocessor).fit_transform(X)
print("Clustering matrix:", X_cluster.shape)

k_range = range(2, 11)
inertia, silhouette = [], []
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X_cluster)
    inertia.append(km.inertia_)
    silhouette.append(silhouette_score(X_cluster, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(k_range), inertia, "o-")
axes[0].set_title("Elbow method")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")
axes[1].plot(list(k_range), silhouette, "o-")
axes[1].set_title("Silhouette score")
axes[1].set_xlabel("k")
plt.show()

In [ ]:
# Choose k after looking at the elbow / silhouette plots above.
# Business-friendly segmentation usually works best with 3-5 groups.
N_CLUSTERS = 4

kmeans = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=SEED).fit(X_cluster)
kmeans_labels = kmeans.labels_

# Hierarchical clustering (Ward linkage)
Z = linkage(X_cluster, method="ward")
plt.figure(figsize=(12, 5))
dendrogram(Z, truncate_mode="lastp", p=20, leaf_rotation=90, show_contracted=True)
plt.title("Hierarchical clustering dendrogram (Ward, truncated)")
plt.xlabel("Cluster size / sample")
plt.ylabel("Distance")
plt.show()

hier_labels = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage="ward").fit_predict(X_cluster)

print("Silhouette  K-Means     :", round(silhouette_score(X_cluster, kmeans_labels), 3))
print("Silhouette  Hierarchical:", round(silhouette_score(X_cluster, hier_labels), 3))
print("Agreement (Adjusted Rand Index) K-Means vs Hierarchical:",
      round(adjusted_rand_score(kmeans_labels, hier_labels), 3))

In [ ]:
# PCA -> 2D for visualisation
pca_full = PCA(random_state=SEED).fit(X_cluster)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)
print("Components needed for 90% variance:", int(np.argmax(cum_var >= 0.90)) + 1)

pca = PCA(n_components=2, random_state=SEED)
pca_coords = pca.fit_transform(X_cluster)
print("Variance explained by PC1 + PC2:", round(pca.explained_variance_ratio_.sum(), 3))

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
sns.scatterplot(x=pca_coords[:, 0], y=pca_coords[:, 1], hue=kmeans_labels,
                palette="tab10", alpha=0.7, ax=axes[0])
axes[0].set_title("K-Means clusters (PCA space)")
sns.scatterplot(x=pca_coords[:, 0], y=pca_coords[:, 1], hue=hier_labels,
                palette="tab10", alpha=0.7, ax=axes[1])
axes[1].set_title("Hierarchical clusters (PCA space)")
sns.scatterplot(x=pca_coords[:, 0], y=pca_coords[:, 1], hue=employee_df["Attrition"],
                palette={0: "tab:blue", 1: "tab:red"}, alpha=0.6, ax=axes[2])
axes[2].set_title("Actual attrition (PCA space)")
for ax in axes:
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
plt.tight_layout()
plt.show()

In [ ]:
# Cluster characteristics + attrition within each cluster
profile_cols = ["Age", "MonthlyIncome", "JobLevel", "YearsAtCompany",
                "TotalWorkingYears", "DistanceFromHome", "JobSatisfaction",
                "OverTime", "Attrition"]

cluster_profile = (
    employee_df.assign(Cluster=kmeans_labels)
    .groupby("Cluster")[profile_cols].mean()
    .rename(columns={"Attrition": "AttritionRate", "OverTime": "OverTimeShare"})
)
cluster_profile.insert(0, "Size", pd.Series(kmeans_labels).value_counts().sort_index())
display(cluster_profile.round(2))

cluster_profile["AttritionRate"].plot.bar(figsize=(7, 4), color="tab:red")
plt.title("Attrition rate by K-Means cluster")
plt.ylabel("Attrition rate")
plt.xticks(rotation=0)
plt.show()

> Note: `OverTimeShare` is only a true share if OverTime is coded 0/1 - check your encoding.
> Give each cluster a business name (e.g. "Early-career, low pay") once you've read the profile.

## 9. Employee risk prediction
To score **all** employees fairly, probabilities come from cross-validation
(*out-of-fold*): every employee is scored by a model that never saw them.
Scoring the training rows with the fitted model would give overconfident numbers.
These are model-generated risk estimates, not claims that an employee will leave.

In [ ]:
oof_proba = cross_val_predict(clone(final_model), X, y, cv=cv,
                              method="predict_proba", n_jobs=-1)[:, 1]

def risk_category(p):
    if p < 0.30:
        return "Low"
    elif p <= 0.60:
        return "Medium"
    return "High"

risk_df = pd.DataFrame({
    "EmployeeID": np.arange(1, len(employee_df) + 1),
    "Predicted_Attrition": (oof_proba >= 0.5).astype(int),
    "Attrition_Probability": oof_proba.round(4),
})
risk_df["Risk_Category"] = risk_df["Attrition_Probability"].apply(risk_category)
risk_df["Risk_Category"] = pd.Categorical(risk_df["Risk_Category"],
                                          ["Low", "Medium", "High"], ordered=True)

# Sanity check: actual attrition should rise from Low -> High
check = (risk_df.assign(Actual=y.values)
         .groupby("Risk_Category", observed=True)
         .agg(Employees=("EmployeeID", "count"), Actual_Attrition_Rate=("Actual", "mean")))
display(check.round(3))

sns.countplot(data=risk_df, x="Risk_Category", order=["Low", "Medium", "High"])
plt.title("Employees by predicted risk category")
plt.show()

> With SMOTE / balanced training the predicted probabilities run higher than the true base rate
> (~16%), so the 30% / 60% cut-offs will label more people Medium/High than the raw attrition rate suggests.
> Treat the categories as a *ranking*; the sanity table above shows whether that ranking works.

## 10. Export for Power BI

In [ ]:
import joblib

EXPORT_DIR = "powerbi_exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

# The cleaned CSV is label-encoded, so Power BI would show Department = 1/2/3.
# OPTIONAL: point this to the ORIGINAL IBM csv (same 1470 rows, same order) to
# add readable text labels for the dashboards. Leave as None to skip.
raw_path = None   # e.g. r"C:\\...\\WA_Fn-UseC_-HR-Employee-Attrition.csv"

employee_export = employee_df.copy()
employee_export.insert(0, "EmployeeID", np.arange(1, len(employee_df) + 1))
employee_export = employee_export.join(risk_df.drop(columns="EmployeeID"))
employee_export["KMeans_Cluster"] = kmeans_labels
employee_export["Hierarchical_Cluster"] = hier_labels
employee_export["PCA1"] = pca_coords[:, 0]
employee_export["PCA2"] = pca_coords[:, 1]

if raw_path:
    raw = pd.read_csv(raw_path)
    assert len(raw) == len(employee_df), "Row count differs - can't align labels safely"
    for col in ["Department", "JobRole", "OverTime", "Gender", "MaritalStatus",
                "EducationField", "BusinessTravel"]:
        employee_export[f"{col}_Label"] = raw[col].values

employee_export.to_csv(f"{EXPORT_DIR}/employee_risk_scores.csv", index=False)

imbalance_df.to_csv(f"{EXPORT_DIR}/imbalance_comparison.csv", index=False)
cv_df.to_csv(f"{EXPORT_DIR}/cross_validation_results.csv", index=False)
tuned_df.assign(Best_Params=tuned_df["Best_Params"].astype(str)).to_csv(
    f"{EXPORT_DIR}/tuned_model_results.csv", index=False)
regression_df.to_csv(f"{EXPORT_DIR}/regression_results.csv", index=False)
cluster_profile.reset_index().to_csv(f"{EXPORT_DIR}/cluster_profile.csv", index=False)

joblib.dump(final_model, f"{EXPORT_DIR}/final_attrition_model.joblib")

print("Saved to:", os.path.abspath(EXPORT_DIR))
print(sorted(os.listdir(EXPORT_DIR)))